In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!ls /content/drive/MyDrive

Mounted at /content/drive
 20191216_122821.mp4		  gptneo-finetuned-qa
 20200101_074036.mp4		  math
 20200324_211649.mp4		 'Screenshot_20220307-174728_().jpg'
 Classroom			  대마도여행250224.gmap
'Colab Notebooks'		  여수여행_241216.gmap
 Colab_Notebooks		 '역사 보고서.show'
'Gantt Chart_07의 사본.gslides'   오키나와여행_241230.gmap
 gptneo-1.3B-finetuned		  통지서.html
 gptneo-1.3B-masked		  홍콩여행_240121.gmap


In [ ]:
# 1. 필요한 패키지 설치
!pip install fastapi pyngrok uvicorn fsspec==2025.3.0 transformers datasets accelerate nest_asyncio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# 2. 모델 로딩 (학습된 모델)
from transformers import GPTNeoForCausalLM, GPT2Tokenizer, pipeline
import torch, json

model_path = "/content/drive/MyDrive/gptneo-1.3B-masked"  # fine-tuned 모델 경로
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
model = GPTNeoForCausalLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# 3. 문제 생성용 기본 모델 로딩 (optional, GPT-Neo base or 다른 사전학습 모델)
base_model_name = "EleutherAI/gpt-neo-1.3B"
base_tokenizer = GPT2Tokenizer.from_pretrained(base_model_name)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = GPTNeoForCausalLM.from_pretrained(base_model_name).to("cuda" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

In [ ]:
# 5. 문제 및 답 생성 함수 정의
def generate_problem(model, tokenizer, topic, max_new_tokens=50, temperature=0.8):
    prompt = f"Generate a new {topic} problem.\nQuestion:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    question = generated_text.split("Question:")[-1].strip().split("\n")[0]
    return f"Question: {question}"

def generate_answer(model, tokenizer, question, max_new_tokens=50, temperature=0.8):
    prompt = f"{question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Answer:" in full_output:
        answer = full_output.split("Answer:")[-1].strip()
    else:
        answer = full_output.strip()
    return answer

In [ ]:
# 6. FastAPI app 설정
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class ProblemRequest(BaseModel):
    topic: str
    count: int

class ProblemResponse(BaseModel):
    response: str

@app.post("/generate", response_model=ProblemResponse)
def generate_problems(req: ProblemRequest):
    problems = []
    for _ in range(req.count):
        # 문제 생성 (base 모델 사용)
        question = generate_problem(base_model, base_tokenizer, req.topic)
        # 정답 생성 (fine-tuned 모델 사용)
        answer = generate_answer(model, tokenizer, question)
        problems.append({"Question": question.replace("Question: ", ""), "Answer": answer})

    return ProblemResponse(
        response=json.dumps(problems, ensure_ascii=False, indent=2)
    )

In [ ]:
# 7. Run ngrok + uvicorn server (Colab only)
from pyngrok import ngrok
import nest_asyncio
import uvicorn

ngrok.set_auth_token("2v1Fi5CEzLumREBpheNMIIepRlM_7uLFbq5PGe81hmEZiAe9K")
ngrok.kill()

public_url = ngrok.connect(3000)
print("🔗 Public URL:", public_url.public_url)

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=3000)

INFO:     Started server process [1214]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:3000 (Press CTRL+C to quit)


🔗 Public URL: https://de2b-35-229-214-226.ngrok-free.app
INFO:     155.230.84.111:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     155.230.84.111:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     155.230.84.111:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     155.230.84.111:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     155.230.84.111:0 - "GET /openapi.json HTTP/1.1" 200 OK


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


INFO:     155.230.84.111:0 - "POST /generate HTTP/1.1" 200 OK
